# 04 — Objective v3: Variable Number of Picks (Phase 0)

So far every algorithm searched for a **fixed** number of picks (K = 10)
and maximized the **mean** semblance of those picks. Letting K be free
does not work with that objective: the mean is maximized by a single
pick on the strongest semblance peak, and a plain sum is maximized by
as many picks as the constraints allow.

**Formulation v3** uses a threshold instead:

$$f(s) = \sum_{k=1}^{K} \big(S(t_k, v_k) - \tau\big), \qquad K \text{ free}$$

A pick only improves the solution if its semblance is above $\tau$:
picks on reflections add, picks on noise subtract.

**Blind formulation.** v3 uses no information taken from the velocity
model (the "answer"):
- **time window**: the whole record (no water-bottom `t_min`, no
  `t_max = 2700 ms`) — picks in the water layer or in noise should be
  rejected by the threshold itself, not forbidden by a constraint
- **velocity range**: 1400–5000 m/s (water velocity as the floor,
  generous ceiling) instead of 1400–3000 m/s
- **threshold**: computed from the semblance panel alone,

$$\tau = \mathrm{median}(P) + c \cdot \sigma_{rob}, \qquad \sigma_{rob} = 1.4826 \cdot \mathrm{MAD}(P)$$

  The semblance ridge covers a small fraction of the panel, so the
  median and the MAD describe the incoherent background; $c$ is the
  number of robust spreads above it.

**Phase 0 runs no optimizer.** It checks whether the objective rewards
the right thing *before* any algorithm is asked to maximize it. The
ground truth (true V_rms, water-bottom time) is used **only to
validate** — it never enters the objective.

**Execution order:**
1. Setup
2. Parameters
3. Load the test cases
4. Blind step: semblance panels and thresholds
5. Validation step: semblance along the true V_rms
6. Visualize the main case
7. Visualize all cases
8. Validation summary
9. Save results
10. Summary


## 1. Setup

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')


In [ ]:
from src.seismic.cases     import load_case, list_cases
from src.seismic.semblance import (compute_semblance, semblance_along_curve,
                                   robust_threshold)
from src.analysis.events   import select_events

print('All modules imported successfully.')


## 2. Parameters

In [ ]:
# --- Test cases (the bank in cases/*.npz) ---
CASES_DIR    = os.path.join(project_root, 'cases')
MAIN_CASE    = 'marmousi2_cdp06800'

# --- Blind formulation v3 (no information from the velocity model) ---
VEL_MIN      = 1400.0        # water velocity as the floor (general knowledge)
VEL_MAX      = 5000.0        # generous ceiling (was 3000, chosen from the
                              # true V_rms of the Coursework 1 case)
VEL_STEP     = 20.0          # velocity increment of the panel (m/s)
MIN_SEP_MS   = 100.0         # minimum separation between picks: ~3 periods
                              # of the 30 Hz wavelet (acquisition knowledge)
# Time window: the whole record — no water-bottom t_min, no t_max.

# --- Threshold rule ---
TAU_FACTORS  = [5.0, 10.0, 15.0]   # c in tau = median + c * sigma_rob
TAU_DEFAULT  = 10.0                # factor used in the detailed figures

# --- Validation only (uses the ground truth) ---
CORRECT_TOL      = 0.02      # relative velocity error of a "correct" velocity
WATER_BOTTOM_GAP = MIN_SEP_MS
                              # ms above the water bottom treated as part of the
                              # water-bottom reflection: the 30 ms semblance gate
                              # plus the wavelet spread its energy upwards, and a
                              # pick closer than MIN_SEP_MS to that event could not
                              # coexist with a pick on the event itself anyway
EARLY_MS         = 200.0     # early times, where shallow trajectories cross
                              # deep events on the far traces

print('Parameters set.')


## 3. Load the test cases

In [ ]:
cases = [load_case(p) for p in list_cases(CASES_DIR)]

for c in cases:
    print(f"{c['name']:28s} gather {c['gather'].shape}  dt={c['dt_ms']:g} ms  "
          f"record={c['t_grid'][-1]:.0f} ms  noise={100 * c['meta']['noise_level']:.0f}%")


## 4. Blind step: semblance panels and thresholds

Everything in this section uses **only the gather**: the panel over the
v3 velocity range and the whole record, and the thresholds computed
from it.

In [ ]:
velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)

for c in cases:
    c['panel'] = compute_semblance(c['gather'], c['offsets'], velocities, c['dt_ms'])
    c['tau']   = {f: robust_threshold(c['panel'], factor=f) for f in TAU_FACTORS}

print(f'{"Case":28s} {"median":>7} {"sigma_rob":>10} ' +
      ' '.join(f'{"tau(c=" + format(f, "g") + ")":>10}' for f in TAU_FACTORS))
print('-' * (48 + 11 * len(TAU_FACTORS)))
for c in cases:
    med = np.median(c['panel'])
    sig = 1.4826 * np.median(np.abs(c['panel'] - med))
    print(f"{c['name']:28s} {med:>7.3f} {sig:>10.4f} " +
          ' '.join(f"{c['tau'][f]:>10.3f}" for f in TAU_FACTORS))


## 5. Validation step: semblance along the true V_rms

From here on the ground truth is used, **only to check** the blind
objective:

- **Natural K** — local maxima of $S(t, v_{true}(t))$ above $\tau$, at
  least `MIN_SEP_MS` apart: the events an ideal velocity function would
  pick, i.e. how many picks the v3 optimum should have.
- **Argmax accuracy** — at the times where the panel rises above
  $\tau$, how often the velocity of maximum semblance is within 2% of
  the true one. Low values would mean the threshold lets through ridges
  at wrong velocities (e.g. multiples — absent in these synthetic data).
- **Water layer** — the largest semblance inside the water layer,
  excluding the water-bottom reflection itself (`WATER_BOTTOM_GAP`) and
  the early times (`EARLY_MS`). If it stays below $\tau$, picks in the
  water are penalized by the objective itself.
- **Early times** — the largest semblance before `EARLY_MS` and its
  velocity. Shallow trajectories cross deep events on the far traces
  (in real processing, an NMO stretch mute removes them); a pick there
  at a high velocity is also blocked by monotonicity, since every
  later pick would need an even higher velocity.

In [ ]:
for c in cases:
    dt, t = c['dt_ms'], c['t_grid']
    c['s_true'] = semblance_along_curve(c['gather'], c['offsets'],
                                        c['v_rms_true'], dt)
    min_sep  = int(round(MIN_SEP_MS / dt))
    vstar    = velocities[np.argmax(c['panel'], axis=0)]
    smax     = c['panel'].max(axis=0)
    rel_err  = np.abs(vstar - c['v_rms_true']) / c['v_rms_true']
    wb       = c['meta']['water_bottom_ms']

    c['events'], c['argmax_ok'] = {}, {}
    for f in TAU_FACTORS:
        tau   = c['tau'][f]
        c['events'][f] = select_events(c['s_true'], tau, min_sep)
        above = smax > tau
        c['argmax_ok'][f] = float(np.mean(rel_err[above] <= CORRECT_TOL)) if above.any() else np.nan

    water = (t >= EARLY_MS) & (t < wb - WATER_BOTTOM_GAP)
    early = t < EARLY_MS
    if water.any():
        wi, wj = np.unravel_index(np.argmax(c['panel'][:, water]), c['panel'][:, water].shape)
        c['water_max'] = float(c['panel'][wi, water][wj])
        c['water_v']   = float(velocities[wi])
        c['water_err'] = float(abs(velocities[wi] - c['v_rms_true'][water][wj])
                               / c['v_rms_true'][water][wj])
    else:
        c['water_max'] = c['water_v'] = c['water_err'] = np.nan
    i, j = np.unravel_index(np.argmax(c['panel'][:, early]), c['panel'][:, early].shape)
    c['early_max'], c['early_v'] = float(c['panel'][i, early][j]), float(velocities[i])

print('Validation done.')


## 6. Visualize the main case

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'figures'), exist_ok=True)

c   = next(c for c in cases if c['name'] == MAIN_CASE)
t   = c['t_grid']
tau = c['tau'][TAU_DEFAULT]
ev  = c['events'][TAU_DEFAULT]

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True,
                         gridspec_kw={'width_ratios': [3, 2]})

axes[0].imshow(c['panel'].T, aspect='auto', cmap='jet', vmin=0, vmax=1,
               extent=[velocities[0], velocities[-1], t[-1], t[0]])
axes[0].contour(velocities, t, c['panel'].T, levels=[tau],
                colors='white', linewidths=0.6)
axes[0].plot(c['v_rms_true'], t, 'w--', linewidth=1.4, label='True V RMS')
axes[0].plot(c['v_rms_true'][ev], t[ev], 'ko', markersize=6,
             markeredgecolor='white', label=f'Natural events (K={len(ev)})')
axes[0].set_xlabel('Velocity (m/s)')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title(f'Panel 1400–5000 m/s, whole record — white contour: tau = {tau:.3f}')
axes[0].legend(loc='lower left')

styles = ['--', '-', ':']
axes[1].plot(c['s_true'], t, color='black', linewidth=1.0,
             label='S along true V RMS')
for f, ls in zip(TAU_FACTORS, styles):
    axes[1].axvline(c['tau'][f], color='firebrick', linestyle=ls, linewidth=1.2,
                    label=f'tau (c={f:g}) = {c["tau"][f]:.3f}')
axes[1].plot(c['s_true'][ev], t[ev], 'o', color='firebrick', markersize=6)
axes[1].axhline(c['meta']['water_bottom_ms'], color='gray', linewidth=0.8,
                linestyle='-.', label='Water bottom (validation only)')
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Semblance')
axes[1].set_title('Semblance along the true V RMS')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Objective v3 (blind) — {MAIN_CASE}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures',
                         '04_objective_v3_main_case.png'), dpi=150)
plt.show()
print('Figure saved.')


## 7. Visualize all cases

In [ ]:
fig, axes = plt.subplots(1, len(cases), figsize=(2.6 * len(cases), 7), sharey=True)

for ax, c in zip(axes, cases):
    t  = c['t_grid']
    ev = c['events'][TAU_DEFAULT]
    ax.plot(c['s_true'], t, color='black', linewidth=0.8)
    for f, ls in zip(TAU_FACTORS, styles):
        ax.axvline(c['tau'][f], color='firebrick', linestyle=ls, linewidth=1.0)
    ax.plot(c['s_true'][ev], t[ev], 'o', color='firebrick', markersize=4)
    ax.axhline(c['meta']['water_bottom_ms'], color='gray', linewidth=0.8, linestyle='-.')
    ax.set_xlim(0, 1)
    ax.set_ylim(t[-1], t[0])
    ax.set_title(f"{c['name'].replace('marmousi2_', '')}\nK = {len(ev)}", fontsize=10)
    ax.set_xlabel('Semblance')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Time (ms)')
plt.suptitle(f'Semblance along the true V RMS — red lines: tau for c = '
             f'{", ".join(format(f, "g") for f in TAU_FACTORS)} '
             f'(dashed, solid, dotted); dots: natural events at c = {TAU_DEFAULT:g}',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures',
                         '04_objective_v3_all_cases.png'), dpi=150)
plt.show()
print('Figure saved.')


## 8. Validation summary

In [ ]:
print('Natural K (events along the true V RMS above tau)')
print(f'{"Case":28s} ' + ' '.join(f'{"c=" + format(f, "g"):>7}' for f in TAU_FACTORS))
print('-' * (29 + 8 * len(TAU_FACTORS)))
for c in cases:
    print(f"{c['name']:28s} " + ' '.join(f"{len(c['events'][f]):>7}" for f in TAU_FACTORS))

print()
print('Argmax accuracy (times above tau whose max-semblance velocity is within 2%)')
print(f'{"Case":28s} ' + ' '.join(f'{"c=" + format(f, "g"):>7}' for f in TAU_FACTORS))
print('-' * (29 + 8 * len(TAU_FACTORS)))
for c in cases:
    print(f"{c['name']:28s} " + ' '.join(f"{c['argmax_ok'][f]:>7.0%}" for f in TAU_FACTORS))

print()
print(f'Water layer and early times (tau at c = {TAU_DEFAULT:g})')
print(f'{"Case":28s} {"tau":>6} {"water max":>10} {"below tau":>10} '
      f'{"at v":>6} {"v err":>6} {"early max":>10} {"at v":>6}')
print('-' * 90)
for c in cases:
    tau = c['tau'][TAU_DEFAULT]
    print(f"{c['name']:28s} {tau:>6.3f} {c['water_max']:>10.3f} "
          f"{'yes' if c['water_max'] < tau else 'NO':>10} "
          f"{c['water_v']:>6.0f} {c['water_err']:>6.1%} "
          f"{c['early_max']:>10.3f} {c['early_v']:>6.0f}")
print()
print('"v err": error of the water-layer maximum against the true V RMS. A small')
print('error means the maximum belongs to the water-bottom reflection seen through')
print('the far traces (hyperbolas with the same velocity converge at far offsets),')
print('not to a false event.')


## 9. Save results

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'metrics'), exist_ok=True)

summary = {
    'formulation': {
        'vel_min': VEL_MIN, 'vel_max': VEL_MAX, 'vel_step': VEL_STEP,
        'min_sep_ms': MIN_SEP_MS, 'time_window': 'whole record',
        'tau_rule': 'median + c * 1.4826 * MAD of the panel',
        'tau_factors': TAU_FACTORS,
    },
    'cases': [{
        'name':        c['name'],
        'tau':         {format(f, 'g'): c['tau'][f] for f in TAU_FACTORS},
        'natural_K':   {format(f, 'g'): int(len(c['events'][f])) for f in TAU_FACTORS},
        'event_times_ms': {format(f, 'g'): (c['events'][f] * c['dt_ms']).tolist()
                           for f in TAU_FACTORS},
        'argmax_accuracy': {format(f, 'g'): c['argmax_ok'][f] for f in TAU_FACTORS},
        'water_max':   c['water_max'],
        'water_max_velocity': c['water_v'],
        'water_max_rel_error': c['water_err'],
        'early_max':   c['early_max'],
        'early_max_velocity': c['early_v'],
    } for c in cases],
}

path = os.path.join(project_root, 'results', 'metrics', 'objective_v3_phase0.json')
with open(path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved: {path}')


## 10. Summary

**What phase 0 checks.** Whether the blind v3 objective rewards the
real reflections and penalizes noise, before any optimizer is run:
- the threshold comes from the panel alone (median + c robust spreads);
- along the true V_rms, the events above $\tau$ give the number of
  picks the v3 optimum should have (natural K);
- inside the water layer, semblance should stay below $\tau$, so picks
  there lower the objective without any water-bottom constraint.

**Limitations to keep in mind.**
- These are idealized synthetic data (primaries only, no multiples,
  generated with the same hyperbolic operator the semblance assumes):
  the argmax accuracy is high partly because there are no false ridges.
- In the clean 3-layer model, the very strong water-bottom reflection
  still shows above $\tau$ just above the water bottom, at the correct
  velocity: hyperbolas with the same velocity converge on the far
  traces. It is the real event seen early, not a false one.
- Early times show coherent-looking values from shallow trajectories
  crossing deep events on the far traces; monotonicity blocks picks
  there at high velocities, and an NMO stretch mute would remove them.
- The factor c is a formulation parameter: results of the optimizers
  under v3 should be reported with its sensitivity.

**Next step (phase 1).** Adapt the optimizers to v3 — variable-length
solutions, insert/remove moves, threshold objective — and run Random
Search, Hill Climbing and Tabu Search again under this formulation,
scored by precision (picks that are correct) and recall (natural
events that received a correct pick).
